# Chapter 02-05 · Distributions, outliers, and transformations

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** moderate - one judgement that never becomes automatic

**Prerequisites:** 02-04. You should know the three missingness mechanisms and why `dropna()` is a
selection filter.

**Position in the learning path:** module 02, chapter 5 of 8. Before: **02-04**. After: **02-06**,
on looking at two and three variables at once.

---

## Why this matters

The last chapter was about values that are **wrong**. This one is about values that are **right and
surprising** - and the difference is a judgement no code can make for you.

Somebody will offer you a rule. *Remove anything more than three standard deviations from the mean.*
It is one line, it feels rigorous, and in this chapter it deletes every single one of the busiest
days of the year - the festival days, when the stand runs out of bikes and Maria loses the most
money. The model's accuracy on ordinary days is unchanged. Its error on festival days goes from 8
bikes to 258.

An outlier is not a category of value. It is a **question**: *why is this row different?* And there
are three possible answers, each demanding a different response.

## What you will be able to do

By the end of this chapter you can:

1. **Read** a distribution: centre, spread, skew, tails, and whether it is one population or
   several.
2. **Choose** between mean, median and mode, and say what each one describes.
3. **Apply** a log transform, say what it does to a relationship, and name three things it costs.
4. **Classify** an outlier as an error, a rare-but-real case, or a different population - and act
   accordingly.
5. **Demonstrate** why automatic outlier removal can leave a model unable to predict the cases that
   matter most.

## Warm-up: retrieve, do not reread

From memory:

1. What is the one question that separates MCAR, MAR and MNAR?
2. Why did mean imputation leave the bias unchanged?
3. What is a sentinel, and why is it worse than a null?
4. Name the cheapest defect check there is.

<br>

*Answers: (1) does the chance of being missing depend on the value that is missing? (2) inserting
the observed mean into a column cannot change that column's mean. (3) a real-looking number standing
in for "not recorded"; `isna()` cannot see it and it sorts and averages as data. (4) `describe()` -
compare min and max against what the world allows.*

## The situation

Nine hundred days at Maria's stand. Most days are ordinary - somewhere between 80 and 140 rentals,
depending mainly on temperature. A few days a year the park holds a festival, and demand goes
through the roof.

Maria wants two things: a number that describes a typical day, and a model that tells her how many
bikes to bring out.

**The question this chapter answers:** what does the shape of this data allow you to say - and what
happens when you tidy away the parts that do not fit?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(17)
n_days = 900

temperature = rng.normal(18, 6, n_days)
festival = rng.random(n_days) < 0.04                          # about one day in 25
ordinary_demand = 40 + 3.5 * temperature + rng.normal(0, 12, n_days)
rentals = np.clip(ordinary_demand + festival * 260, 5, None).round()   # SYNTHETIC

days = pd.DataFrame({"temp_c": temperature.round(1),
                     "festival": festival.astype(int),
                     "rentals": rentals})
print(days["rentals"].describe().round(1).to_string())

### Reading a distribution

`describe()` gives you six numbers. Read them in this order:

1. **count** - is it the number of rows you expect?
2. **min and max** - are they possible? (02-04's cheapest check)
3. **mean versus 50% (the median)** - if they differ, the distribution is skewed, and the direction
   tells you which way.
4. **the quartiles** - 25% to 75% is where the middle half of the data lives. Compare that width
   with the distance from 75% to max.

Here the mean is **110.8** and the median is **104.0**. The mean is higher, which means a right
(positive) skew: a tail of large values pulling the average up. And the max, 400-odd, is far beyond
the 75th percentile.

**Skew** puts a number on it. Zero means symmetric; positive means a long right tail.

In [ ]:
print(f"mean   {days['rentals'].mean():.1f}")
print(f"median {days['rentals'].median():.1f}")
print(f"mode   {days['rentals'].round(-1).mode().iloc[0]:.0f}  (to the nearest ten)")
print(f"skew   {days['rentals'].skew():.2f}")
print()
print(f"middle half of days: {days['rentals'].quantile(0.25):.0f} to {days['rentals'].quantile(0.75):.0f}")
print(f"top 1% of days     : above {days['rentals'].quantile(0.99):.0f}")

In [ ]:
fig, (raw, logged) = plt.subplots(1, 2, figsize=(11, 3.8))

raw.hist(days["rentals"], bins=45, color="#0072B2", edgecolor="white")
raw.axvline(days["rentals"].mean(), color="#D55E00", linewidth=2, label=f"mean {days['rentals'].mean():.0f}")
raw.axvline(days["rentals"].median(), color="#009E73", linewidth=2, linestyle="--",
            label=f"median {days['rentals'].median():.0f}")
raw.set_xlabel("Rentals per day (count)"); raw.set_ylabel("Number of days")
raw.set_title("As measured: one long right tail")
raw.legend(fontsize=8)

logged.hist(np.log10(days["rentals"]), bins=45, color="#0072B2", edgecolor="white")
logged.set_xlabel("log10(rentals per day)"); logged.set_ylabel("Number of days")
logged.set_title("On a log scale: two humps become visible")

fig.tight_layout()
plt.show()

### Mean, median, mode - three different questions

| Summary | Answers | Sensitive to |
|---|---|---|
| **Mean** | If I shared the total equally across all days, how much per day? | Every value, including the extremes |
| **Median** | What does the middle day look like? | Only the ordering |
| **Mode** | What is the most common outcome? | The peak, not the tail |

The mean of 110.8 describes **no day that happened**: it is above 60% of days and far below the
festival days. That is not a flaw in the mean - it is answering the total-divided-by-count question
faithfully - but if Maria asks "what is a normal day like?", the median of 104 is the honest answer.

**The rule:** on a skewed distribution, report both, and say which question each one answers. If the
two agree, one number will do.

And look at the right-hand histogram. On a log scale the single long tail resolves into **two humps**
- a large one for ordinary days and a small separate one far to the right. That is the shape of two
populations sharing a column, and it is the first hint that the festival days are not merely large,
they are *different*.

## The log transform

Taking logs turns **multiplication into addition** and pulls in a long right tail. It is the most
useful transformation in applied work, and it is worth knowing exactly what it does and what it
costs.

In [ ]:
values = np.array([10, 100, 1000, 10000])
print("values      :", values)
print("log10       :", np.log10(values))
print("equal ratios become equal distances: 10x -> +1 every time")
print()
print(f"raw skew    : {days['rentals'].skew():.2f}")
print(f"log10 skew  : {np.log10(days['rentals']).skew():.2f}")

Skew falls from **3.69 to 1.12**. The tail is compressed, and a difference of "ten times bigger" now
occupies the same distance wherever it appears on the scale.

**When a log transform is the right move:**

- The quantity is **positive and multiplicative** - income, prices, populations, counts, durations,
  areas. Doubling means the same thing at every level.
- The **relative** error is what matters. Being 10 out on a value of 20 is serious; being 10 out on
  a value of 4,000 is not. Modelling `log(y)` makes the error proportional.
- A relationship is **curved on raw axes and straight on log axes**, which is the usual sign that
  something multiplicative is going on.

**Three things it costs:**

1. **Zeros and negatives are undefined.** `log(0)` is negative infinity. The usual patch is
   `log1p(x)` (which computes `log(1 + x)`), and it is a patch - it changes the shape near zero,
   and if a third of your data is zero the transform is not the answer.
2. **Interpretability.** A coefficient on `log(income)` is a statement about percentage changes, not
   euros. You must be able to say what it means out loud, or you should not use it.
3. **The units change, and so does what your metric optimises.** Minimising error on `log(y)` is not
   minimising error on `y` - it weights small values much more heavily. And converting a prediction
   back with `exp()` does **not** give you the mean of `y`; it gives approximately the *median*.
   Chapter 05-04 covers this properly, and it surprises people in production.

**The honest summary:** a log transform is a change of question, not a cleaning step. Apply it when
the multiplicative version of the question is the one you want answered.

---

## Failure lab: the three-sigma rule

Here is the rule somebody will offer you.

> *Anything more than three standard deviations from the mean is an outlier. Remove it.*

It sounds principled. It comes from the normal distribution, where 99.7% of values fall inside three
standard deviations, so anything outside is genuinely rare. Applied to this data, it removes 29 of
900 rows - about 3%, which sounds like careful housekeeping.

### Predict before running

1. Which 29 days will it remove?
2. What will happen to the model's error on ordinary days?
3. What will happen to its error on festival days?

In [ ]:
mean, sd = days["rentals"].mean(), days["rentals"].std()
inside = (days["rentals"] - mean).abs() <= 3 * sd

print(f"three-sigma cut-off: {mean - 3 * sd:.0f} to {mean + 3 * sd:.0f} rentals")
print(f"rows removed       : {(~inside).sum()} of {len(days)} ({(~inside).mean():.1%})")
print(f"of those removed   : {days.loc[~inside, 'festival'].sum()} are festival days")
print(f"festival days kept : {days.loc[inside, 'festival'].sum()} of {days['festival'].sum()}")

**All 29 removed rows are festival days, and every single festival day in the dataset is removed.**

The rule did not find a handful of anomalies. It found *the entire second population* and deleted
it - because that population is exactly what "far from the mean" means when the mean was computed
over both.

Now train two models and see what it cost.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

features = ["temp_c", "festival"]
train, test = np.arange(n_days) < 700, np.arange(n_days) >= 700
X, y = days[features].to_numpy(), days["rentals"].to_numpy()

def evaluate(keep_mask, label):
    fit_rows = train & keep_mask
    model = LinearRegression().fit(X[fit_rows], y[fit_rows])
    predicted = model.predict(X[test])
    is_festival = festival[test]
    print(f"{label:<26} all days {mean_absolute_error(y[test], predicted):6.1f} | "
          f"ordinary {mean_absolute_error(y[test][~is_festival], predicted[~is_festival]):5.1f} | "
          f"festival {mean_absolute_error(y[test][is_festival], predicted[is_festival]):7.1f}")

evaluate(np.ones(n_days, dtype=bool), "keep everything")
evaluate(inside.to_numpy(), "three-sigma removal")
print(f"\n(the test set contains {festival[test].sum()} festival days out of {test.sum()})")

### Diagnosis

| | All days | Ordinary days | Festival days |
|---|---|---|---|
| Keep everything | **9.5** | 9.6 | **8.3** |
| Three-sigma removal | 19.5 | 9.6 | **258.4** |

**Error on ordinary days: identical.** The removal changed nothing for the 97% of days that were
never at risk.

**Error on festival days: 8.3 becomes 258.4** - thirty times worse. The model has never seen a
festival, so it predicts an ordinary day and is short by roughly 250 bikes.

**And the overall error doubles**, from 9.5 to 19.5, entirely from those few days.

**Why this is the worst possible trade.** The festival days are the days Maria most needs a
prediction for. On an ordinary day, being 10 bikes out costs a little. On a festival day, being 250
short means turning away hundreds of customers, and there are only a few such days a year to get
right. **The rule removed the rows with the highest cost of error and left the ones with the lowest.**

**Why the rule failed, mechanically.** Three-sigma assumes a normal distribution. This data is not
normal - it is skewed with a second population - so the standard deviation is inflated by the very
points being tested, and "three sigma" no longer means "rare". The rule is a statement about a shape
this data does not have.

**And note what the removal did *not* break.** The model that never saw a festival is not broken;
it is a perfectly good model of ordinary days. It simply has no idea that festival days exist, and
it says so nowhere.

### The three kinds of outlier

An outlier is a question. There are exactly three answers, and they demand different actions.

| Kind | What it is | Example here | What to do |
|---|---|---|---|
| **An error** | The value is wrong | Age 200, a negative count, a decimal-point slip | Fix it, or set it to missing. Never silently keep it |
| **Rare but real** | The value is right and unusual | A genuinely exceptional hot day | **Keep it.** It is data. Consider a robust method or a transform |
| **A different population** | The row belongs to another process | A festival day | **Keep it and label it.** Add a feature, or model the groups separately |

**The festival days are the third kind**, and the third kind has a specific right answer: *give the
model the flag*. We already had `festival` as a column - it is in the features - and that is why
"keep everything" predicts festival days to within 8 bikes. The information was there. Removing the
rows threw it away.

### Remedies

| Remedy | What it does | When |
|---|---|---|
| Look at the rows before deleting any | Turns "outlier" into "festival day" | Always. It takes a minute |
| Add a feature that identifies the group | Lets the model handle both populations | When the group is identifiable - usually |
| Use quantiles, not sigmas (e.g. below the 1st or above the 99th percentile) | Does not assume normality | When you need a threshold at all |
| Use a robust model or a transform | Reduces the influence of extremes without deleting them | When the extremes are noise you must tolerate |
| Model the segments separately | Two populations, two models | When they differ in kind, not just in level |
| Report error by segment, never one number | Makes this failure visible immediately | Always - 07-05 is built on it |

The last row is what would have caught this in one line. A single overall MAE of 19.5 looks
acceptable. **Broken down by segment it is obviously catastrophic**, and the breakdown costs one
`groupby`.

## Common misconceptions

**"Three standard deviations is the standard definition of an outlier."**
It is a rule of thumb for *normal* data. On a skewed distribution the standard deviation is inflated
by the tail, so the rule is being applied with a ruler the data bent.

**"Outliers should be removed before modelling."**
Errors should be corrected. Real extremes are data. Which one you have is a question about the
world, and the answer is often that the extremes are the most valuable rows you own - the fraud, the
churn, the festival, the failure.

**"A skewed distribution needs to be made normal."**
Very few methods in this course require normally distributed *features* or *targets*. Transform
because it makes the relationship simpler or the error measure more appropriate - not to satisfy an
assumption nobody checked whether you had.

**"The mean is the average, so it is the summary."**
The mean answers "total divided by count". On skewed data that describes no individual case. Say
which question you are answering.

**"Two humps in a histogram just means I need more bins."**
Sometimes. More often it means two populations, and finding the variable that separates them is more
valuable than any amount of modelling on the mixture.

**"Log transforms make the data better."**
They make it *different*. Zeros break, units change, coefficients become percentages, and
back-transforming a prediction gives you a median rather than a mean. All fine if intended, all
surprising if not.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-05_distributions_solutions.ipynb`.

### Quick understanding

**E1 (define).** Name the three kinds of outlier and the right response to each.

**E2 (explain).** Why did the three-sigma rule remove exactly the festival days, rather than a
scattering of unusual days?

**E3 (explain).** The mean is 110.8 and the median is 104.0. What does each one answer, and which
would you give Maria if she asked what a normal day looks like?

### Hand calculation

**E4 (calculate).** Seven days of rentals: 90, 95, 100, 105, 110, 115, 400. Compute the mean, the
median, and the mean after removing the 400. Then compute the standard deviation with and without
the 400. What does the second calculation tell you about applying a three-sigma rule to this data?

**E5 (calculate).** A quantity doubles every step: 5, 10, 20, 40, 80, 160. Compute the differences
between consecutive values, then the differences between their base-2 logarithms. What has the log
transform done, and what would a straight line on log axes mean?

### Coding

**E6 (code).** Write `distribution_report(series)` printing count, mean, median, mode, skew, the
quartiles, the 1st and 99th percentiles, and the share of values beyond three standard deviations
and beyond the 99th percentile. Run it on `rentals` and on `log10(rentals)`.

**E7 (code).** Replace the three-sigma rule with a quantile rule (drop below the 1st and above the
99th percentile) and repeat the failure lab. Does it help? Report error by segment.

### Interpretation

**E8 (interpret).** The overall MAE after outlier removal is 19.5 and on ordinary days it is 9.6. A
colleague says "19.5 is fine, we're within 20 bikes". Explain what is wrong with that sentence in
terms of what the model will be used for.

### Debugging

**E9 (diagnose).** A revenue model performs well on most customers and badly on the largest accounts,
which are a small fraction of customers and most of the revenue. Give three explanations rooted in
this chapter, and say what you would look at first.

### Exam and interview reasoning

**E10 (defend).** *"How do you handle outliers?"* Answer in about 130 words. Naming a threshold rule
loses.

**E11 (design).** Maria wants a single number for her weekly report describing "a typical day", plus
a warning system for unusual days. Design both: which summary, which threshold, and what the warning
should say.

### Transfer to a different situation

**E12 (design).** For each, say which kind of outlier it probably is and what you would do:
(a) a patient recorded as 700 kg; (b) a transaction of 40,000 EUR in a dataset averaging 60 EUR;
(c) a server response time of 30 seconds when the median is 80 ms; (d) a house price of 1 EUR;
(e) a sensor reading of exactly 0.00 on 4% of days.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why removing the "unusual" days made the
forecast worse, not better.

### Optional challenge

**E14 (code + diagnose).** Compare four treatments of the festival days on the same test set: keep
everything with the flag, keep everything without the flag (drop the `festival` column), remove them
by three-sigma, and fit two separate models. Report error by segment for each and say which you
would ship and why.

In [ ]:
# Your workspace. Still in memory: days, rentals, temperature, festival, inside, X, y, train, test.

## Mastery check

Without scrolling up, can you:

- [ ] Name the three kinds of outlier and the response to each? *(If not: "The three kinds".)*
- [ ] Say why three-sigma failed on this data? *(If not: "Diagnosis".)*
- [ ] Say what the mean, median and mode each answer? *(If not: "Three different questions".)*
- [ ] Name three costs of a log transform? *(If not: "The log transform".)*
- [ ] Say the one-line check that makes this failure visible? *(If not: the remedies table.)*

## What should now feel instinctive

1. **"Why is this row different?"** - asked before any row is deleted.
2. **Compare the mean with the median.** If they disagree, the distribution is skewed and one number
   will mislead.
3. **Two humps means two populations.** Go and find the variable that separates them.
4. **Report error by segment, never one number.** It is one `groupby` and it catches this class of
   failure immediately.
5. **The rows with the biggest errors are usually the rows that matter most.** Deleting them is the
   most expensive kind of tidying.

## Flashcards

| Question | Answer |
|---|---|
| The three kinds of outlier | An error; rare but real; a different population |
| What to do with each | Fix or null it; keep it; keep it and label it |
| Why did three-sigma delete every festival? | It assumes normality, and the standard deviation is inflated by the very points being tested |
| Mean vs median vs mode | Total divided by count; the middle case; the most common case |
| What does a log transform do? | Turns equal ratios into equal distances - multiplication into addition |
| Three costs of logs | Zeros and negatives undefined; coefficients become percentages; back-transforming gives a median, not a mean |
| When does a log help? | Positive multiplicative quantities, and when relative error is what matters |
| Two humps in a histogram | Usually two populations sharing a column |
| Why is skew a problem for the mean? | The mean is pulled by the tail and may describe no case that occurred |
| The one-line check that catches bad outlier handling | Report the error by segment, not as a single number |

## Next

**02-06 · Univariate, bivariate, multivariate: looking without fooling yourself.**

So far every chapter has looked at one column at a time. The moment you look at two, a new class of
mistake becomes available: relationships that appear, disappear or reverse depending on what else
you hold constant. The festival days were a second population hiding inside one column; the next
chapter is about the structure you can only see by looking at columns *together* - and the paradox
that has embarrassed more analysts than any other.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).